## algo for single tree detection 
- high of the buildings
- high of the trees
- computitaions including buildings in the viewshed - no higher buildigns on the "line"

- from where do we compute the view? 
- do we count floors? so person can see the tree form different floor? 

- count by street adress??

- data: tree polygons from detection
- street adresses - civico from 300m 
- buildings: global buildings atlas


The possible workflow:

- DeepForest on RGB → tree crown polygons 
- LiDAR CHM (DSM − DTM) → height raster
- zonal_stats(tree_polygons, chm_raster, stats='max') → height per tree

for each civico c:
    building = footprint_lookup(c)
    observers = generate_floor_observers(building)  # Option B
    
    visible_trees = set()
    for obs in observers:
        for tree in trees_within_radius(obs, max_radius=300m):
            if los_clear(obs, tree, dsm):
                visible_trees.add(tree.id)
    
    result[c] = {
        'civico': c,
        'n_visible_trees': len(visible_trees),
        'satisfies_rule': len(visible_trees) >= 3
    }

Phase 1 — Data prep
├── Acquire LiDAR DSM, DTM → compute CHM
├── Run DeepForest on RGB orthophoto → crown polygons GeoDataFrame
└── zonal_stats(crowns, CHM) → tree_height column

Phase 2 — Address → building join
├── Spatially join civici points → building footprints
└── Compute floor observers per building (height / 3.0, min 1 floor)

Phase 3 — LOS computation  (this is the expensive step)
├── For each observer point: query candidate trees within 150m
├── Ray march through DSM raster along each O→T line
└── Flag tree as visible if no DSM cell exceeds LOS height

Phase 4 — Aggregate
├── Union visible trees across all floors of building
├── Count unique visible tree IDs per civico
└── Output GeoDataFrame with civico + n_visible_trees + satisfies_rule